In [25]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import pandas as pd
from typing import List
from data_loader import fetch_categories_mmlu, load_mmlu_dataset
import json

## Loading the MMLU dataset
normative_category = [
    "moral_disputes",
    "philosophy",
    "world_religions",
    "us_foreign_policy",
    "sociology",
    "professional_psychology",
    "professional_law",
    "moral_scenarios",
    "human_sexuality",
    "international_law",
]

control_category = ["college_mathematics",
    "college_physics",
    "formal_logic",
    "logical_fallacies",
    "college_computer_science",
]

dataset_subjects = normative_category + control_category

mmlu_df = fetch_categories_mmlu(dataset_subjects)

samples_per_subject = 50
samples_examples_per_subject = 5

sample_mmlu, sample_examples_mmlu = load_mmlu_dataset(
    mmlu_df,
    sample_per_subject=samples_per_subject,
    sample_examples_per_subject=samples_examples_per_subject,
    random_state=42,
    random_state_examples=0)

print(f"Sample MMLU shape: {sample_mmlu.shape}")
print(f"Sample Examples MMLU shape: {sample_examples_mmlu.shape}")
print("Subjects in sample_mmlu:", sample_mmlu["subject"].nunique())
print("Subjects in sample_examples_mmlu:", sample_examples_mmlu["subject"].nunique())



### Loading the full set of samples: 
sample_mmlu_full = pd.read_csv("data/samples_mmlu_full.csv", index_col=False)

print(f"\n\nFull sample MMLU shape: {sample_mmlu_full.shape}")
print("Subjects in sample_mmlu_full:", sample_mmlu_full["subject"].nunique())
print("List of subjects in sample_mmlu_full: ", sample_mmlu_full["subject"].unique())

Sample MMLU shape: (750, 7)
Sample Examples MMLU shape: (75, 7)
Subjects in sample_mmlu: 15
Subjects in sample_examples_mmlu: 15


Full sample MMLU shape: (579, 7)
Subjects in sample_mmlu_full: 4
List of subjects in sample_mmlu_full:  ['college_mathematics' 'moral_scenarios' 'professional_law' 'formal_logic']


In [27]:
from dotenv import load_dotenv
import openai
import os, torch, numpy as np

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_AZURE_OPENAI": "Azure OpenAI",
    "ENDPOINT_AZURE_OPENAI": "Azure OpenAI Endpoint",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")

client = openai.OpenAI(api_key=os.getenv("API_KEY_OPENAI"))
model = "gpt-4.1-mini"
model_filename = "openai_4.1_mini"

# Zero-shot and Few Shot on MMLU

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields


### Use this variable to test on 4 categories with more samples (from 9)
MMLU_DATA_FULL = True

ZERO_SHOT = False # Set to false for Few Shot
nb_examples_few_shots = 3
    
case = mmlu_case
case_name = case.case_name

if MMLU_DATA_FULL:
    
    data = sample_mmlu_full

    if ZERO_SHOT:

        output_file = f"results/{model_filename}/zero_shot/classic/results_mmlu_zero_shot_full.csv"
        classifier = ZeroShot(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
        )

    else:
        output_file = f"results/{model_filename}/few_shot/classic/results_mmlu_few_shot_{nb_examples_few_shots}_shot_full.csv"
        classifier = FewShot(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
            n_shots=nb_examples_few_shots,
            examples_df=sample_examples_mmlu,
        )

else:
    data = sample_mmlu

    if ZERO_SHOT:
        output_file = f"results/{model_filename}/zero_shot/classic/results_mmlu_zero_shot.csv"
        classifier = ZeroShot(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
        )
    else:
        output_file = f"results/{model_filename}/few_shot/classic/results_mmlu_few_shot_{nb_examples_few_shots}_shot.csv"
        classifier = FewShot(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
            n_shots=nb_examples_few_shots,
            examples_df=sample_examples_mmlu,
        )


os.makedirs(os.path.dirname(output_file), exist_ok=True)

if os.path.exists(output_file):
    df_out_existing = pd.read_csv(output_file)
    done_ids = set(df_out_existing["sample_id"])
    rows = df_out_existing.to_dict(orient="records")
    print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
else:
    done_ids = set()
    rows = []


try:
    for idx, row in tqdm(data.iterrows(), total=len(data)):
        if idx in done_ids:
            continue

        text = row[case.input_col]
        true_label = row[case.label_col]
        if isinstance(true_label, str):
            true_label = true_label.strip()

        try:
            predicted_label, stats = classifier.classify(text)
            mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
            
        except RateLimitError as e:
            print(f"\nRate limit hit at sample {idx}. Saving progress.")
            break
        except Exception as e:
            print(f"\nError at sample {idx}: {e}. Skipping.")
            continue

        additional = get_additional_fields(row, case_name) 

        results = {
            "sample_id": idx,
            "text": text,
            "true_label": true_label,
            "pred_label": mapped_label,
            "max_tokens": classifier.max_tokens,
            "tokens_used": stats["tokens_used"],
            "prompt_tokens": stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "latency": stats["latency"],
            **additional,
        }

        rows.append(results)

except KeyboardInterrupt:
    print("=== Interrupted manually. Saving progress...")

finally:
    df_out = pd.DataFrame(rows)
    df_out.to_csv(output_file, index=False)
    print(f"✅ Saved {len(df_out)} rows to {output_file}")
    
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)

    print("=== Classification Report ===\n")
    print(classification_report(y_true, y_pred))

    print("\n=== Confusion Matrix ===\n")
    labels = sorted(set(y_true) | set(y_pred))
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

    accuracy = (y_true == y_pred).mean()
    print(f"\n=== Accuracy: {accuracy:.2%} ===")

## Role-playing in Zero-shot and Few-shots settings (3 examples)

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields
from profiles.profile_sets import PERSON_ETHNICS

selected_profiles = [f"profile{i}" for i in range(1, 61)]

### Use this variable to test on 4 categories with more samples (from 9)
MMLU_DATA_FULL = True

ZERO_SHOT = False # Set to false for Few Shot

role_playing = "passive"

person_set = PERSON_ETHNICS

case = mmlu_case
case_name = case.case_name
nb_examples_few_shots = 3



for person_key in selected_profiles:
    if MMLU_DATA_FULL:
        data = sample_mmlu_full

        if ZERO_SHOT:
            output_file_suffix = f"zero_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_zero_shot_full.csv"
            classifier = ZeroShot(
                case=case,
                client=client,
                model=model,
                max_tokens=300,
                person_key=person_key,
                role_playing=role_playing,
                person_set=person_set
            )
            loop_desc = f"mmlu | {person_key} | zero-shot | {role_playing}"
            
        else:
            output_file_suffix = f"few_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_few_shot_3examples_full.csv"
            classifier = FewShot(
                case=case,
                client=client,
                model=model,
                max_tokens=300,
                n_shots=3,
                examples_df=sample_examples_mmlu,
                person_key=person_key,
                role_playing=role_playing,
                person_set=person_set
            )
            loop_desc = f"mmlu | {person_key} | few-shot | {role_playing}"
    else:
        data = sample_mmlu
   
        if ZERO_SHOT:
            output_file_suffix = f"zero_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_zero_shot_full.csv"
            classifier = ZeroShot(
                case=case,
                client=client,
                model=model,
                max_tokens=300,
                person_key=person_key,
                role_playing=role_playing,
                person_set=person_set
            )
            loop_desc = f"mmlu | {person_key} | zero-shot | {role_playing}"
        else:
            output_file_suffix = f"few_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_few_shot_3examples_full.csv"
            classifier = FewShot(
                case=case,
                client=client,
                model=model,
                max_tokens=300,
                n_shots=3,
                examples_df=sample_examples_mmlu,
                person_key=person_key,
                role_playing=role_playing,
                person_set=person_set
            )
            loop_desc = f"mmlu | {person_key} | few-shot | {role_playing}"


    output_file = f"results/{model_filename}/" + output_file_suffix
    
    #os.makedirs(os.path.dirname(output_file), exist_ok=True)

    #if os.path.exists(output_file):
    #    df_out_existing = pd.read_csv(output_file)
    #    done_ids = set(df_out_existing["sample_id"])
    #    rows = df_out_existing.to_dict(orient="records")
    #    print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
    #else:
    done_ids = set()
    rows = []

    try:
        for idx, row in tqdm(data.iterrows(), total=len(data), desc=loop_desc):
            if idx in done_ids:
                continue

            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()

            try:
                predicted_label, stats = classifier.classify(text)
                mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
            except RateLimitError as e:
                print(f"Error : {e}")
                print(f"WARNING: Rate limit hit at sample {idx}. Skipping.")
                continue
            except Exception as e:
                print(f"ERROR at sample {idx}: {e}")
                continue

            additional = get_additional_fields(row, case_name)

            results = {
                "sample_id": idx,
                "text": text,
                "true_label": true_label,
                "pred_label": mapped_label,
                "max_tokens": classifier.max_tokens,
                "tokens_used": stats["tokens_used"],
                "prompt_tokens": stats["prompt_tokens"],
                "completion_tokens": stats["completion_tokens"],
                "latency": stats["latency"],
                **additional,
            }

            rows.append(results)

    except KeyboardInterrupt:
        print("=== Interrupted manually.")

    df_out = pd.DataFrame(rows)
    if len(df_out) == 579:
        df_out.to_csv(output_file, index=False)
    print(f"✅ Saved {len(df_out)} rows to {output_file}")
    
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)
    
    print("=== Classification Report ===\n")
    print(classification_report(y_true, y_pred))
    
    print("\n=== Confusion Matrix ===\n")
    labels = sorted(set(y_true) | set(y_pred))
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
    
    accuracy = (y_true == y_pred).mean()
    print(f"\n=== Accuracy for {loop_desc}: {accuracy:.2%} ===")

mmlu | profile56 | few-shot | passive: 100%|██████████| 579/579 [06:46<00:00,  1.42it/s]

✅ Saved 579 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile56_passive/results_mmlu_few_shot_3examples_full.cs
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.62      0.51      0.56       150
           1       0.55      0.65      0.59       125
           2       0.64      0.67      0.65       138
           3       0.71      0.70      0.71       166

    accuracy                           0.63       579
   macro avg       0.63      0.63      0.63       579
weighted avg       0.63      0.63      0.63       579


=== Confusion Matrix ===

    0   1   2    3
0  76  40  21   13
1  13  81  13   18
2  15  14  92   17
3  19  13  17  117

=== Accuracy for mmlu | profile56 | few-shot | passive: 63.21% ===


# Chain of Thought - MMLU dataset

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from chain_of_thought import ChainOfThoughts
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields


### Use this variable to test on 4 categories with more samples (from 9)
MMLU_DATA_FULL = True
    
case = mmlu_case
case_name = case.case_name
max_tokens = 700

strategy = "optimized"          # "optimized", "zero_plus"


if MMLU_DATA_FULL:
    data = sample_mmlu_full
else:
    data = sample_mmlu



#output_file = f"results/{model_filename}/cot/classic/results_mmlu__optimized_cot.csv"

classifier = ChainOfThoughts(
        case=case,
        client=client,
        model=model,
        max_tokens=max_tokens,
    )

os.makedirs(os.path.dirname(output_file), exist_ok=True)

done_ids = set()
rows = []
detailed_reasoning = []

for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
    text = row[case.input_col]
    true_label = row[case.label_col]

    if isinstance(true_label, str):
        true_label = true_label.strip()

    try:
        predicted_label, metrics = classifier.classify_with_strategy(
            text, strategy=strategy
        )

        mapped_label = case.label_map.get(
            predicted_label.strip(), list(case.label_map.values())[-1]
        )

        results = {
            "sample_id": idx,
            "text": text,
            "true_label": true_label,
            "pred_label": mapped_label,
            "raw_pred_label": predicted_label,
            "max_tokens": classifier.max_tokens,
            "tokens_used": metrics.get("tokens_used"),
            "prompt_tokens": metrics.get("prompt_tokens"),
            "completion_tokens": metrics.get("completion_tokens"),
            "latency": metrics.get("latency"),
            "strategy": strategy,
        }
        rows.append(results)

        raw_resp = metrics.get("raw_response", "")
        steps, analysis, final_line = classifier._parse_steps_and_final(raw_resp)
        reasoning_detail = {
            "sample_id": idx,
            "raw_response": raw_resp,
            "parsed_steps": steps,
            "analysis": analysis,       
            "final_line": final_line,
            "final_label": predicted_label,
            "mapped_label": mapped_label,
        }
        detailed_reasoning.append(reasoning_detail)

    except Exception as e:
        print(f"Error processing sample {idx}: {e}")
        continue

if not rows:
    print(f"No successful classifications for {case_name}")

output_file = f"results/{model_filename}/cot/classic/results_mmlu_{strategy}_cot.csv"
reasoning_file = f"results/{model_filename}/cot/classic/reasoning_mmlu_{strategy}_cot.json"

os.makedirs(os.path.dirname(output_file), exist_ok=True)

df_out = pd.DataFrame(rows)
df_out.to_csv(output_file, index=False)
print(f"=== Saved {len(df_out)} rows to {output_file} ===")

with open(reasoning_file, 'w', encoding='utf-8') as f:
    json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
    print(f"=== Saved detailed reasoning to {reasoning_file} ===")

try:
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)

    print(f"\n=== Classification Report for {case_name} ({strategy}) ===")
    print(classification_report(y_true, y_pred, zero_division=0))
    print(f"\n=== Confusion Matrix for {case_name} ===")
    labels = sorted(set(y_true) | set(y_pred))
    print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))

    accuracy = (y_true == y_pred).mean()
    print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")

    print(f"\n=== Label Distribution ===")
    print("True labels:")
    print(pd.Series(y_true).value_counts())
    print("Predicted labels:")
    print(pd.Series(y_pred).value_counts())

except Exception as e:
    print(f"Error in evaluation for {case_name}: {e}")

metrics = cot_classifier.get_metrics()
print(f"\n=== CoT Performance Metrics ===")
print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
print(f"- Total calls: {metrics['total_calls']}")


print(f"\n=== Sample Reasoning (raw/parsed) ===")
for i, reasoning in enumerate(detailed_reasoning[:3]):
    print(f"\nSample {reasoning['sample_id']}:")
    print(f"True Label: {data.loc[reasoning['sample_id']][case.label_col]}")
    print(f"Predicted: {reasoning['final_label']} -> Mapped: {reasoning['mapped_label']}")
    
    if reasoning['parsed_steps']:
        print("Parsed Steps:")
        for step in reasoning['parsed_steps']:
            print(f"  Step {step['step']}: {step['content'][:150]}...")
    else:
        print("No explicit 'Step N' blocks found.")
    if reasoning['final_line']:
        print(f"Final Line: {reasoning['final_line']}")
    print("Raw Response (truncated):")
    print((reasoning['raw_response'] or "")[:300] + "...")
    print("-" * 50)

print("\n=== Chain of Thought Evaluation Complete ===")

# Chain of Thought - Role Playing with MMLU dataset

In [ ]:
from tqdm import tqdm
import os, json, re
import pandas as pd
from chain_of_thought import ChainOfThoughts
from sklearn.metrics import classification_report, confusion_matrix
from cases.mmlu_case import mmlu_case

from profiles.profile_sets import PERSON_ETHNICS

### Use this variable to test on 4 categories with more samples (from 9)
MMLU_DATA_FULL = True
    
case = mmlu_case
case_name = case.case_name
max_tokens = 700

strategy = "optimized"          # "optimized", "zero_plus"


if MMLU_DATA_FULL:
    data = sample_mmlu_full
else:
    data = sample_mmlu


role_playing_mode = "passive"

selected_profiles = ["profile1"]

try:
    for profile_n in selected_profiles:
        print(f"\n=== Running Chain of Thought for {case_name} ({strategy}) - {profile_n} ===")

        cot_classifier = ChainOfThoughts(
            case=case,
            client=client,
            model=model,
            max_tokens=max_tokens,
            person_key=profile_n,
            role_playing=role_playing_mode, 
            person_set=PERSON_ETHNICS
        )

        rows = []
        detailed_reasoning = []

        for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()

            try:
                predicted_label, metrics = cot_classifier.classify_with_strategy(
                    text, strategy=strategy
                )

                mapped_label = case.label_map.get(
                    predicted_label.strip(), list(case.label_map.values())[-1]
                )

                results = {
                    "sample_id": idx,
                    "text": text,
                    "true_label": true_label,
                    "pred_label": mapped_label,
                    "raw_pred_label": predicted_label,
                    "max_tokens": cot_classifier.max_tokens,
                    "tokens_used": metrics.get("tokens_used"),
                    "prompt_tokens": metrics.get("prompt_tokens"),
                    "completion_tokens": metrics.get("completion_tokens"),
                    "latency": metrics.get("latency"),
                    "strategy": strategy,
                }
                rows.append(results)

                raw_resp = metrics.get("raw_response", "")
                steps, analysis, final_line = cot_classifier._parse_steps_and_final(raw_resp)
                reasoning_detail = {
                    "sample_id": idx,
                    "raw_response": raw_resp,
                    "parsed_steps": steps,
                    "analysis": analysis,       
                    "final_line": final_line,
                    "final_label": predicted_label,
                    "mapped_label": mapped_label,
                }
                detailed_reasoning.append(reasoning_detail)

            except Exception as e:
                print(f"Error processing sample {idx}: {e}")
                continue

        if not rows:
            print(f"No successful classifications for {case_name}")
            continue

        output_file = f"results/{model_filename}/cot/role_playing_ethnics/{profile_n}_{role_playing_mode}/results_mmlu_{strategy}_cot.csv"
        reasoning_file = f"results/{model_filename}/cot/role_playing_ethnics/{profile_n}_{role_playing_mode}/reasoning_mmlu_{strategy}_cot.json"

        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        os.makedirs(os.path.dirname(reasoning_file), exist_ok=True)

        df_out = pd.DataFrame(rows)
        df_out.to_csv(output_file, index=False)
        print(f"=== Saved {len(df_out)} rows to {output_file} ===")

        with open(reasoning_file, 'w', encoding='utf-8') as f:
            json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
        print(f"=== Saved detailed reasoning to {reasoning_file} ===")


        try:
            if case_name.lower() == "manipulation":
                y_true = df_out["true_label"].astype(int)
                y_pred = df_out["pred_label"].astype(int)
            elif case_name.lower() == "stereotype":
                y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()

            print(f"\n=== Classification Report for {case_name} ({strategy}) ===")
            print(classification_report(y_true, y_pred, zero_division=0))
            print(f"\n=== Confusion Matrix for {case_name} ===")
            labels = sorted(set(y_true) | set(y_pred))
            print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))

            accuracy = (y_true == y_pred).mean()
            print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")

            print(f"\n=== Label Distribution ===")
            print("True labels:")
            print(pd.Series(y_true).value_counts())
            print("Predicted labels:")
            print(pd.Series(y_pred).value_counts())

        except Exception as e:
            print(f"Error in evaluation for {case_name}: {e}")

        metrics = cot_classifier.get_metrics()
        print(f"\n=== CoT Performance Metrics ===")
        print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
        print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
        print(f"- Total calls: {metrics['total_calls']}")


        print(f"\n=== Sample Reasoning (raw/parsed) ===")
        for i, reasoning in enumerate(detailed_reasoning[:3]):
            print(f"\nSample {reasoning['sample_id']}:")
            print(f"True Label: {data.loc[reasoning['sample_id']][case.label_col]}")
            print(f"Predicted: {reasoning['final_label']} -> Mapped: {reasoning['mapped_label']}")
            if reasoning['parsed_steps']:
                print("Parsed Steps:")
                for step in reasoning['parsed_steps']:
                    print(f"  Step {step['step']}: {step['content'][:150]}...")
            else:
                print("No explicit 'Step N' blocks found.")
            if reasoning['final_line']:
                print(f"Final Line: {reasoning['final_line']}")
            print("Raw Response (truncated):")
            print((reasoning['raw_response'] or "")[:300] + "...")
            print("-" * 50)

    
except Exception as e:
    print(f"== Error: {e} ==")
    import traceback
    traceback.print_exc()

print("\n=== Chain of Thought Evaluation Complete ===")